# Phase 1 – Project environment setup

In [1]:
# Cell 1: Verify Python environment

import sys
import os

print("Python Version:", sys.version)
print("Python Executable:", sys.executable)
print("Project Working Directory:", os.getcwd())

Python Version: 3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
Python Executable: c:\Users\ADMIN\anaconda3\python.exe
Project Working Directory: c:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project\python


In [2]:
# Cell 2: Check installed project dependencies

import importlib

required_libraries = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "sqlalchemy",
    "mysql.connector",
    "dotenv",
    "openpyxl"
]

for library in required_libraries:
    try:
        importlib.import_module(library)
        print(f"{library:<20} : Installed")
    except ImportError:
        print(f"{library:<20} : Not Installed")

pandas               : Installed
numpy                : Installed
matplotlib           : Installed
seaborn              : Installed
sqlalchemy           : Installed
mysql.connector      : Installed
dotenv               : Installed
openpyxl             : Installed


In [3]:
# Cell 3: Test Python–MySQL connectivity

import mysql.connector
from getpass import getpass

db_password = getpass("Enter MySQL root password: ")

try:
    connection = mysql.connector.connect(
        host="localhost",
        port=3306,
        user="root",
        password=db_password,
        connection_timeout=10
    )

    if connection.is_connected():
        cursor = connection.cursor()
        cursor.execute("SELECT VERSION()")
        mysql_version = cursor.fetchone()[0]

        print("MySQL Connection: SUCCESS")
        print("MySQL Version:", mysql_version)

        cursor.close()

except mysql.connector.Error as error:
    print("MySQL Connection: FAILED")
    print("Error:", error)

finally:
    if "connection" in locals() and connection.is_connected():
        connection.close()
        print("Connection closed successfully.")

MySQL Connection: SUCCESS
MySQL Version: 8.0.46
Connection closed successfully.


In [4]:
# Cell 4: Create Medallion database and schemas

import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password
)

cursor = connection.cursor()

cursor.execute("""
    CREATE DATABASE IF NOT EXISTS ecommerce_analytics
""")

cursor.execute("""
    USE ecommerce_analytics
""")

schemas = ["bronze", "silver", "gold"]

for schema in schemas:
    cursor.execute(f"""
        CREATE SCHEMA IF NOT EXISTS {schema}
    """)
    print(f"Schema created/verified: {schema}")

connection.commit()

cursor.close()
connection.close()

print("\nMedallion database setup completed.")

Schema created/verified: bronze
Schema created/verified: silver
Schema created/verified: gold

Medallion database setup completed.


In [5]:
# Cell 5: Verify database and schemas

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

cursor.execute("""
    SELECT SCHEMA_NAME
    FROM INFORMATION_SCHEMA.SCHEMATA
    WHERE SCHEMA_NAME IN ('bronze', 'silver', 'gold')
    ORDER BY SCHEMA_NAME
""")

for row in cursor.fetchall():
    print(row[0])

cursor.close()
connection.close()

bronze
gold
silver


In [7]:
# Cell 6: Configure project paths

from pathlib import Path

# Project root directory
PROJECT_ROOT = Path.cwd().parent

# Source dataset
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ecommerce_customer.csv"

print("Project Root:", PROJECT_ROOT)
print("Dataset Path:", DATA_PATH)
print("Dataset Exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    print("Dataset Size:", round(DATA_PATH.stat().st_size / 1024, 2), "KB")
else:
    print("ERROR: Dataset not found. Check the file location.")

Project Root: c:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project
Dataset Path: c:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project\data\raw\ecommerce_customer.csv
Dataset Exists: True
Dataset Size: 601.7 KB


In [8]:
# Cell 8: Load and inspect the source dataset

import pandas as pd

df_raw = pd.read_csv(DATA_PATH)

print("Dataset successfully loaded!")
print("-" * 45)

print("Number of Rows:", df_raw.shape[0])
print("Number of Columns:", df_raw.shape[1])

print("\nColumn Names:")
print(df_raw.columns.tolist())

print("\nFirst 5 Records:")
display(df_raw.head())

Dataset successfully loaded!
---------------------------------------------
Number of Rows: 3055
Number of Columns: 32

Column Names:
['Order_ID', 'Customer_ID', 'Order_Date', 'Customer_Age', 'Customer_Segment', 'City', 'Region', 'Category', 'Product_Name', 'Sales_Channel', 'Payment_Method', 'Device', 'Quantity', 'Unit_Price', 'Discount_Pct', 'Shipping_Days', 'Customer_Rating', 'Delivery_Rating', 'Return_Status', 'Order_Status', 'Marketing_Source', 'Customer_Loyalty_Points', 'Website_Session_Minutes', 'Items_Viewed', 'Previous_Orders', 'Customer_Satisfaction', 'Gross_Amount', 'Discount_Amount', 'Net_Sales', 'Estimated_Cost', 'Profit', 'Profit_Margin_Pct']

First 5 Records:


,Order_ID,Customer_ID,Order_Date,Customer_Age,Customer_Segment,City,Region,Category,Product_Name,Sales_Channel,...,Website_Session_Minutes,Items_Viewed,Previous_Orders,Customer_Satisfaction,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
0,ORD100994,C1229,2025-06-07,44.0,Occasional,Delhi,Central,Sports,Shoes,Online,...,16.4,10,10,1.8,723.34,0.00,723.34,634.67,88.67,12.26
1,ORD101072,C1627,2025-03-18,51.0,Occasional,Kochi,West,Home Appliances,Shoes,Online,...,23.2,9,3,5.0,NaN,NaN,NaN,NaN,NaN,NaN
2,ORD100995,C1410,2025-07-13,NaN,Occasional,Hyderabad,West,Electronics,Laptop,Online,...,9.1,10,6,3.0,739.05,105.68,633.37,370.35,263.02,41.53
3,ORD100731,C1578,2025-09-24,38.0,Regular,Bengaluru,North,Fashion,Shoes,Mobile App,...,22.7,4,4,3.6,28004.44,1473.03,26531.41,17911.62,8619.79,32.49
4,ORD102954,C1524,2025-06-09,39.0,Premium,Kolkata,South,Electronics,Skincare,Marketplace,...,6.6,8,8,2.5,9841.78,1842.38,7999.40,6214.63,1784.77,22.31


In [9]:
# Cell 9: Verify dataset data types and memory usage

print("Dataset Information:")
df_raw.info()

print("\nDataset Memory Usage:")
print(round(df_raw.memory_usage(deep=True).sum() / 1024, 2), "KB")

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 3055 entries, 0 to 3054
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Order_ID                 3055 non-null   str    
 1   Customer_ID              3055 non-null   str    
 2   Order_Date               3055 non-null   str    
 3   Customer_Age             2932 non-null   float64
 4   Customer_Segment         3055 non-null   str    
 5   City                     2979 non-null   str    
 6   Region                   3055 non-null   str    
 7   Category                 3055 non-null   str    
 8   Product_Name             3055 non-null   str    
 9   Sales_Channel            3055 non-null   str    
 10  Payment_Method           2978 non-null   str    
 11  Device                   3055 non-null   str    
 12  Quantity                 3055 non-null   int64  
 13  Unit_Price               2962 non-null   float64
 14  Discount_Pct  

In [10]:
# Cell 10: Complete source dataset profile

source_profile = pd.DataFrame({
    "Column": df_raw.columns,
    "Data_Type": df_raw.dtypes.astype(str).values,
    "Non_Null_Count": df_raw.notna().sum().values,
    "Missing_Count": df_raw.isna().sum().values,
    "Missing_Percentage": (
        df_raw.isna().mean().values * 100
    ).round(2)
})

display(source_profile)

print("\nTotal Records:", len(df_raw))
print("Total Columns:", len(df_raw.columns))
print("Total Missing Values:", df_raw.isna().sum().sum())

,Column,Data_Type,Non_Null_Count,Missing_Count,Missing_Percentage
0,Order_ID,str,3055,0,0.00
1,Customer_ID,str,3055,0,0.00
2,Order_Date,str,3055,0,0.00
3,Customer_Age,float64,2932,123,4.03
4,Customer_Segment,str,3055,0,0.00
5,City,str,2979,76,2.49
6,Region,str,3055,0,0.00
7,Category,str,3055,0,0.00
8,Product_Name,str,3055,0,0.00
9,Sales_Channel,str,3055,0,0.00



Total Records: 3055
Total Columns: 32
Total Missing Values: 2518


In [11]:
# Cell 11: Save source data quality baseline

REPORT_DIR = PROJECT_ROOT / "reports" / "data_quality"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

PROFILE_PATH = REPORT_DIR / "source_data_profile.csv"

source_profile.to_csv(PROFILE_PATH, index=False)

print("Source profiling report saved successfully!")
print("Report Location:", PROFILE_PATH)

Source profiling report saved successfully!
Report Location: c:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project\reports\data_quality\source_data_profile.csv


# Phase 2 – Bronze Layer: Raw Data Ingestion

In [12]:
# Cell 12: Create Bronze raw table

import mysql.connector

create_bronze_table = """
CREATE TABLE IF NOT EXISTS bronze.ecommerce_raw (

    Order_ID TEXT,
    Customer_ID TEXT,
    Order_Date TEXT,
    Customer_Age TEXT,
    Customer_Segment TEXT,
    City TEXT,
    Region TEXT,
    Category TEXT,
    Product_Name TEXT,
    Sales_Channel TEXT,
    Payment_Method TEXT,
    Device TEXT,
    Quantity TEXT,
    Unit_Price TEXT,
    Discount_Pct TEXT,
    Shipping_Days TEXT,
    Customer_Rating TEXT,
    Delivery_Rating TEXT,
    Return_Status TEXT,
    Order_Status TEXT,
    Marketing_Source TEXT,
    Customer_Loyalty_Points TEXT,
    Website_Session_Minutes TEXT,
    Items_Viewed TEXT,
    Previous_Orders TEXT,
    Customer_Satisfaction TEXT,
    Gross_Amount TEXT,
    Discount_Amount TEXT,
    Net_Sales TEXT,
    Estimated_Cost TEXT,
    Profit TEXT,
    Profit_Margin_Pct TEXT,

    ingestion_id BIGINT AUTO_INCREMENT PRIMARY KEY,
    source_file VARCHAR(255),
    ingestion_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP

);
"""

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()
cursor.execute(create_bronze_table)

print("Bronze raw table created or already exists.")

cursor.close()
connection.close()

Bronze raw table created or already exists.


In [13]:
# Cell 13: Verify Bronze table columns

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

cursor.execute("""
    DESCRIBE bronze.ecommerce_raw
""")

for column in cursor.fetchall():
    print(column[0], "|", column[1], "|", column[3])

cursor.close()
connection.close()

Order_ID | text | 
Customer_ID | text | 
Order_Date | text | 
Customer_Age | text | 
Customer_Segment | text | 
City | text | 
Region | text | 
Category | text | 
Product_Name | text | 
Sales_Channel | text | 
Payment_Method | text | 
Device | text | 
Quantity | text | 
Unit_Price | text | 
Discount_Pct | text | 
Shipping_Days | text | 
Customer_Rating | text | 
Delivery_Rating | text | 
Return_Status | text | 
Order_Status | text | 
Marketing_Source | text | 
Customer_Loyalty_Points | text | 
Website_Session_Minutes | text | 
Items_Viewed | text | 
Previous_Orders | text | 
Customer_Satisfaction | text | 
Gross_Amount | text | 
Discount_Amount | text | 
Net_Sales | text | 
Estimated_Cost | text | 
Profit | text | 
Profit_Margin_Pct | text | 
ingestion_id | bigint | PRI
source_file | varchar(255) | 
ingestion_timestamp | datetime | 


In [14]:
# Cell 14: Validate Bronze structure and prepare raw CSV

import pandas as pd
import mysql.connector

# Reload source without automatic type conversion or NA interpretation
df_bronze = pd.read_csv(
    DATA_PATH,
    dtype=str,
    keep_default_na=False
)

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

# Retrieve Bronze table structure
cursor.execute("DESCRIBE bronze.ecommerce_raw")
bronze_columns = [row[0] for row in cursor.fetchall()]

# Check existing records
cursor.execute("SELECT COUNT(*) FROM bronze.ecommerce_raw")
existing_records = cursor.fetchone()[0]

cursor.close()
connection.close()

print("Source columns:", len(df_bronze.columns))
print("Bronze table columns:", len(bronze_columns))
print("Bronze metadata columns:", len(bronze_columns) - len(df_bronze.columns))

print("\nSource and Bronze column names match:",
      list(df_bronze.columns) == bronze_columns[:32])

print("Source records:", len(df_bronze))
print("Existing Bronze records:", existing_records)

print("\nRaw data types:")
print(df_bronze.dtypes.value_counts())

print("\nRaw CSV preview:")
display(df_bronze.head(3))

Source columns: 32
Bronze table columns: 35
Bronze metadata columns: 3

Source and Bronze column names match: True
Source records: 3055
Existing Bronze records: 0

Raw data types:
str    32
Name: count, dtype: int64

Raw CSV preview:


,Order_ID,Customer_ID,Order_Date,Customer_Age,Customer_Segment,City,Region,Category,Product_Name,Sales_Channel,...,Website_Session_Minutes,Items_Viewed,Previous_Orders,Customer_Satisfaction,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
0,ORD100994,C1229,2025-06-07,44.0,Occasional,Delhi,Central,Sports,Shoes,Online,...,16.4,10,10,1.8,723.34,0.0,723.34,634.67,88.67,12.26
1,ORD101072,C1627,2025-03-18,51.0,Occasional,Kochi,West,Home Appliances,Shoes,Online,...,23.2,9,3,5.0,,,,,,
2,ORD100995,C1410,2025-07-13,,Occasional,Hyderabad,West,Electronics,Laptop,Online,...,9.1,10,6,3.0,739.05,105.68,633.37,370.35,263.02,41.53


In [15]:
# Cell 15: Bronze layer - Raw data ingestion

import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

try:
    # Prevent accidental duplicate ingestion
    cursor.execute("SELECT COUNT(*) FROM bronze.ecommerce_raw")
    existing_records = cursor.fetchone()[0]

    if existing_records > 0:
        raise Exception(
            f"Bronze table already contains {existing_records} records. "
            "Ingestion cancelled."
        )

    # Prepare source columns
    source_columns = list(df_bronze.columns)

    insert_columns = source_columns + ["source_file"]

    column_names = ", ".join(
        f"`{column}`" for column in insert_columns
    )

    placeholders = ", ".join(["%s"] * len(insert_columns))

    insert_query = f"""
        INSERT INTO bronze.ecommerce_raw ({column_names})
        VALUES ({placeholders})
    """

    # Prepare records with ingestion metadata
    records = [
        tuple(row) + (DATA_PATH.name,)
        for row in df_bronze.itertuples(index=False, name=None)
    ]

    # Batch insertion
    batch_size = 500

    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        cursor.executemany(insert_query, batch)

        print(
            f"Inserted batch: "
            f"{start + 1} - {start + len(batch)}"
        )

    connection.commit()

    print("\nBronze ingestion completed successfully!")
    print("Total records inserted:", cursor.rowcount)

except Exception as error:
    connection.rollback()
    print("\nIngestion failed. Transaction rolled back.")
    print("Error:", error)

finally:
    cursor.close()
    connection.close()

Inserted batch: 1 - 500
Inserted batch: 501 - 1000
Inserted batch: 1001 - 1500
Inserted batch: 1501 - 2000
Inserted batch: 2001 - 2500
Inserted batch: 2501 - 3000
Inserted batch: 3001 - 3055

Bronze ingestion completed successfully!
Total records inserted: 55


In [16]:
# Cell 16: Verify Bronze ingestion

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

# Total records in Bronze
cursor.execute("""
    SELECT COUNT(*)
    FROM bronze.ecommerce_raw
""")

bronze_count = cursor.fetchone()[0]

# Verify source file metadata
cursor.execute("""
    SELECT
        source_file,
        COUNT(*) AS record_count,
        COUNT(DISTINCT ingestion_id) AS unique_ingestion_ids
    FROM bronze.ecommerce_raw
    GROUP BY source_file
""")

metadata_result = cursor.fetchall()

print("Source Records:", len(df_bronze))
print("Bronze Records:", bronze_count)
print("Record Count Match:", len(df_bronze) == bronze_count)

print("\nIngestion Metadata:")
for row in metadata_result:
    print(row)

cursor.close()
connection.close()

Source Records: 3055
Bronze Records: 3055
Record Count Match: True

Ingestion Metadata:
('ecommerce_customer.csv', 3055, 3055)


In [17]:
# Cell 17: Exact source-to-Bronze comparison

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

source_columns = list(df_bronze.columns)

column_names = ", ".join(
    f"`{column}`" for column in source_columns
)

cursor.execute(f"""
    SELECT {column_names}
    FROM bronze.ecommerce_raw
    ORDER BY ingestion_id
""")

bronze_records = cursor.fetchall()

cursor.close()
connection.close()

# Reconstruct Bronze DataFrame
df_bronze_db = pd.DataFrame(
    bronze_records,
    columns=source_columns
)

# Compare source and Bronze values
comparison = df_bronze.reset_index(drop=True).eq(
    df_bronze_db.reset_index(drop=True)
)

# Count mismatches
mismatch_count = int((~comparison).sum().sum())

column_mismatches = (~comparison).sum()

print("Source Records:", len(df_bronze))
print("Bronze Records:", len(df_bronze_db))
print("Total Columns Compared:", len(source_columns))
print("Total Values Compared:", df_bronze.size)
print("Total Mismatched Values:", mismatch_count)

print("\nExact Data Match:", mismatch_count == 0)

print("\nColumn-wise Mismatch Summary:")
display(column_mismatches[column_mismatches > 0])

Source Records: 3055
Bronze Records: 3055
Total Columns Compared: 32
Total Values Compared: 97760
Total Mismatched Values: 0

Exact Data Match: True

Column-wise Mismatch Summary:


Series([], dtype: int64)

In [18]:
# Cell 18: Bronze layer audit report

bronze_audit = {
    "Source_File": DATA_PATH.name,
    "Source_Records": len(df_bronze),
    "Bronze_Records": len(df_bronze_db),
    "Source_Columns": len(df_bronze.columns),
    "Total_Values_Compared": df_bronze.size,
    "Mismatched_Values": mismatch_count,
    "Unique_Ingestion_IDs": bronze_count,
    "Record_Count_Match": len(df_bronze) == bronze_count,
    "Exact_Data_Match": mismatch_count == 0,
    "Ingestion_Status": "SUCCESS"
}

bronze_audit_df = pd.DataFrame(
    [bronze_audit]
)

BRONZE_AUDIT_PATH = (
    REPORT_DIR / "bronze_ingestion_audit.csv"
)

bronze_audit_df.to_csv(
    BRONZE_AUDIT_PATH,
    index=False
)

display(bronze_audit_df)

print("\nAudit report saved:", BRONZE_AUDIT_PATH)

,Source_File,Source_Records,Bronze_Records,Source_Columns,Total_Values_Compared,Mismatched_Values,Unique_Ingestion_IDs,Record_Count_Match,Exact_Data_Match,Ingestion_Status
0,ecommerce_customer.csv,3055,3055,32,97760,0,3055,True,True,SUCCESS



Audit report saved: c:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project\reports\data_quality\bronze_ingestion_audit.csv


In [19]:
# Cell 19: Load Bronze data for Silver processing

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

df_silver_source = pd.read_sql(
    "SELECT * FROM bronze.ecommerce_raw",
    connection
)

connection.close()

print("Bronze data loaded successfully!")
print("Rows:", df_silver_source.shape[0])
print("Columns:", df_silver_source.shape[1])

display(df_silver_source.head())

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14864\4341804.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_silver_source = pd.read_sql(


Bronze data loaded successfully!
Rows: 3055
Columns: 35


,Order_ID,Customer_ID,Order_Date,Customer_Age,Customer_Segment,City,Region,Category,Product_Name,Sales_Channel,...,Customer_Satisfaction,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct,ingestion_id,source_file,ingestion_timestamp
0,ORD100994,C1229,2025-06-07,44.0,Occasional,Delhi,Central,Sports,Shoes,Online,...,1.8,723.34,0.0,723.34,634.67,88.67,12.26,1,ecommerce_customer.csv,2026-11-20 08:42:40
1,ORD101072,C1627,2025-03-18,51.0,Occasional,Kochi,West,Home Appliances,Shoes,Online,...,5.0,,,,,,,2,ecommerce_customer.csv,2026-11-20 08:42:40
2,ORD100995,C1410,2025-07-13,,Occasional,Hyderabad,West,Electronics,Laptop,Online,...,3.0,739.05,105.68,633.37,370.35,263.02,41.53,3,ecommerce_customer.csv,2026-11-20 08:42:40
3,ORD100731,C1578,2025-09-24,38.0,Regular,Bengaluru,North,Fashion,Shoes,Mobile App,...,3.6,28004.44,1473.03,26531.41,17911.62,8619.79,32.49,4,ecommerce_customer.csv,2026-11-20 08:42:40
4,ORD102954,C1524,2025-06-09,39.0,Premium,Kolkata,South,Electronics,Skincare,Marketplace,...,2.5,9841.78,1842.38,7999.4,6214.63,1784.77,22.31,5,ecommerce_customer.csv,2026-11-20 08:42:40


# Phase 3 – Silver Layer

In [20]:
# Cell 20: Bronze data quality assessment

from sqlalchemy import create_engine, URL

# Create SQLAlchemy connection
db_url = URL.create(
    drivername="mysql+mysqlconnector",
    username="root",
    password=db_password,
    host="localhost",
    port=3306,
    database="ecommerce_analytics"
)

engine = create_engine(db_url)

# Load persisted Bronze data
df_silver_source = pd.read_sql_query(
    "SELECT * FROM bronze.ecommerce_raw",
    engine
)

# Separate business columns and metadata
business_columns = df_bronze.columns.tolist()

df_business = df_silver_source[business_columns].copy()

# Convert empty strings to missing values for profiling only
df_profile = df_business.replace(r"^\s*$", pd.NA, regex=True)

# Column-level quality profile
bronze_quality_profile = pd.DataFrame({
    "Column": business_columns,
    "Missing_Count": df_profile.isna().sum().values,
    "Missing_Percentage": (
        df_profile.isna().mean().values * 100
    ).round(2),
    "Unique_Values": df_profile.nunique().values
})

# Duplicate analysis
duplicate_order_count = (
    df_business["Order_ID"].duplicated(keep=False).sum()
)

duplicate_record_count = df_business.duplicated().sum()

print("BRONZE DATA QUALITY SUMMARY")
print("-" * 45)

print("Total Records:", len(df_business))
print("Total Columns:", len(df_business.columns))
print("Unique Orders:", df_business["Order_ID"].nunique())
print("Unique Customers:", df_business["Customer_ID"].nunique())

print("Duplicate Order_ID Records:", duplicate_order_count)
print("Duplicate Complete Records:", duplicate_record_count)

print("\nColumn-Level Missing Value Profile:")
display(bronze_quality_profile)

engine.dispose()

BRONZE DATA QUALITY SUMMARY
---------------------------------------------
Total Records: 3055
Total Columns: 32
Unique Orders: 3000
Unique Customers: 776
Duplicate Order_ID Records: 110
Duplicate Complete Records: 50

Column-Level Missing Value Profile:


,Column,Missing_Count,Missing_Percentage,Unique_Values
0,Order_ID,0,0.00,3000
1,Customer_ID,0,0.00,776
2,Order_Date,0,0.00,365
3,Customer_Age,123,4.03,74
4,Customer_Segment,0,0.00,4
5,City,75,2.45,21
6,Region,0,0.00,5
7,Category,0,0.00,16
8,Product_Name,0,0.00,10
9,Sales_Channel,0,0.00,4


In [21]:
# Cell 21: Identify duplicate orders

# Find all records belonging to duplicated Order_IDs
duplicate_orders = df_business[
    df_business["Order_ID"].duplicated(keep=False)
].copy()

# Sort for easier investigation
duplicate_orders = duplicate_orders.sort_values(
    by=["Order_ID", "Customer_ID"]
)

# Count records per Order_ID
duplicate_summary = (
    duplicate_orders.groupby("Order_ID")
    .agg(
        Record_Count=("Order_ID", "size"),
        Unique_Customers=("Customer_ID", "nunique"),
        Unique_Order_Status=("Order_Status", "nunique")
    )
    .reset_index()
)

print("Duplicate Order IDs:", duplicate_summary.shape[0])
print("Records involved:", len(duplicate_orders))

print("\nDuplicate Order Summary:")
display(duplicate_summary.head(20))

print("\nDetailed Duplicate Records:")
display(duplicate_orders.head(20))

Duplicate Order IDs: 55
Records involved: 110

Duplicate Order Summary:


,Order_ID,Record_Count,Unique_Customers,Unique_Order_Status
0,ORD100045,2,1,1
1,ORD100053,2,1,1
2,ORD100252,2,1,1
3,ORD100260,2,1,1
4,ORD100290,2,1,1
5,ORD100292,2,1,1
6,ORD100297,2,1,1
7,ORD100341,2,1,1
8,ORD100409,2,1,1
9,ORD100427,2,1,2



Detailed Duplicate Records:


,Order_ID,Customer_ID,Order_Date,Customer_Age,Customer_Segment,City,Region,Category,Product_Name,Sales_Channel,...,Website_Session_Minutes,Items_Viewed,Previous_Orders,Customer_Satisfaction,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
888,ORD100045,C1152,2025-01-25,34.0,Premium,Bengaluru,South,Home Appliances,Shoes,Online,...,6.0,4,5,3.2,2371.46,29.88,2341.58,1517.05,824.53,35.21
1514,ORD100045,C1152,2025-01-25,34.0,Premium,Bengaluru,South,Home Appliances,Shoes,Online,...,6.0,4,5,3.2,2371.46,29.88,2341.58,1517.05,824.53,35.21
1814,ORD100053,C1602,2025-05-13,30.0,Regular,Bengaluru,North,Furniture,Treadmill,Online,...,21.1,7,4,4.5,18455.28,3532.34,14922.94,8850.48,6072.46,40.69
1964,ORD100053,C1602,2025-05-13,30.0,Regular,Bengaluru,North,Furniture,Treadmill,Online,...,21.1,7,4,4.5,18455.28,3532.34,14922.94,8850.48,6072.46,40.69
1276,ORD100252,C1598,2025-06-30,43.0,Regular,Kolkata,West,Books,Shirt,Online,...,16.0,7,6,4.3,6712.56,36.25,6676.31,5004.13,1672.18,25.05
1678,ORD100252,C1598,2025-06-30,43.0,Regular,Kolkata,West,Books,Shirt,Online,...,16.0,7,6,4.3,6712.56,36.25,6676.31,5004.13,1672.18,25.05
1118,ORD100260,C1329,2025-08-03,36.0,Regular,,West,Sports,Chair,Online,...,19.6,11,8,4.7,2792.16,800.79,1991.37,1646.86,344.51,17.3
2863,ORD100260,C1329,2025-08-03,36.0,Regular,,West,Sports,Chair,Online,...,19.6,11,8,4.7,2792.16,800.79,1991.37,1646.86,344.51,17.3
31,ORD100290,C1149,2025-06-22,17.0,Regular,Kolkata,South,Sports,Skincare,Mobile App,...,11.1,12,5,4.6,2540.22,7.87,2532.35,1967.18,565.17,22.32
2950,ORD100290,C1149,2025-06-22,17.0,Regular,Kolkata,South,Sports,Skincare,Mobile App,...,11.1,12,5,4.6,2540.22,7.87,2532.35,1967.18,565.17,22.32


In [24]:
# Cell 22: Classify exact and conflicting duplicate orders

# Count total records per Order_ID
record_counts = (
    duplicate_orders
    .groupby("Order_ID")
    .size()
    .rename("Record_Count")
)

# Remove identical rows temporarily to identify distinct records
distinct_records = duplicate_orders[
    business_columns
].drop_duplicates()

# Count distinct records per Order_ID
unique_counts = (
    distinct_records
    .groupby("Order_ID")
    .size()
    .rename("Unique_Records")
)

# Combine both counts
duplicate_classification = pd.concat(
    [record_counts, unique_counts],
    axis=1
).reset_index()

# Classify duplicate groups
duplicate_classification["Duplicate_Type"] = (
    duplicate_classification["Unique_Records"]
    .map({
        1: "Exact Duplicate"
    })
    .fillna("Conflicting Duplicate")
)

print("Duplicate Classification Summary:")

display(
    duplicate_classification["Duplicate_Type"]
    .value_counts()
    .rename_axis("Duplicate_Type")
    .reset_index(name="Group_Count")
)

print("\nConflicting Duplicate Orders:")

display(
    duplicate_classification[
        duplicate_classification["Duplicate_Type"]
        == "Conflicting Duplicate"
    ]
)

Duplicate Classification Summary:


,Duplicate_Type,Group_Count
0,Exact Duplicate,50
1,Conflicting Duplicate,5



Conflicting Duplicate Orders:


,Order_ID,Record_Count,Unique_Records,Duplicate_Type
9,ORD100427,2,2,Conflicting Duplicate
13,ORD100700,2,2,Conflicting Duplicate
33,ORD101794,2,2,Conflicting Duplicate
38,ORD102113,2,2,Conflicting Duplicate
40,ORD102116,2,2,Conflicting Duplicate


In [25]:
# Cell 23: Find columns containing conflicting values

conflicting_orders = duplicate_classification.loc[
    duplicate_classification["Duplicate_Type"]
    == "Conflicting Duplicate",
    "Order_ID"
]

conflict_details = []

for order_id in conflicting_orders:

    order_records = duplicate_orders[
        duplicate_orders["Order_ID"] == order_id
    ]

    for column in business_columns:

        unique_values = order_records[column].nunique(
            dropna=False
        )

        if unique_values > 1:

            conflict_details.append({
                "Order_ID": order_id,
                "Column": column,
                "Unique_Values": unique_values,
                "Values": order_records[column].tolist()
            })

conflict_details_df = pd.DataFrame(conflict_details)

display(conflict_details_df)

,Order_ID,Column,Unique_Values,Values
0,ORD100427,Order_Status,2,"[Pending, Delivered]"
1,ORD100700,Order_Status,2,"[Delayed, Delivered]"
2,ORD101794,Order_Status,2,"[Pending, Delivered]"
3,ORD102113,Order_Status,2,"[Returned, Delivered]"
4,ORD102116,Order_Status,2,"[Delivered, Delayed]"


In [26]:
# Cell 24: Verify conflicting records differ only by Order_Status

conflict_columns = []

for order_id in conflicting_orders:

    group = duplicate_orders[
        duplicate_orders["Order_ID"] == order_id
    ]

    different_columns = [
        column
        for column in business_columns
        if group[column].nunique(dropna=False) > 1
    ]

    conflict_columns.append({
        "Order_ID": order_id,
        "Conflicting_Columns": different_columns,
        "Conflict_Count": len(different_columns)
    })

conflict_validation = pd.DataFrame(conflict_columns)

display(conflict_validation)

print(
    "\nAll conflicts limited to Order_Status:",
    conflict_validation["Conflicting_Columns"]
    .apply(lambda cols: cols == ["Order_Status"])
    .all()
)

,Order_ID,Conflicting_Columns,Conflict_Count
0,ORD100427,[Order_Status],1
1,ORD100700,[Order_Status],1
2,ORD101794,[Order_Status],1
3,ORD102113,[Order_Status],1
4,ORD102116,[Order_Status],1



All conflicts limited to Order_Status: True


In [27]:
# Cell 25: Verify conflicting orders directly in MySQL

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

conflict_query = """
SELECT
    Order_ID,
    COUNT(*) AS Record_Count,
    COUNT(DISTINCT Order_Status) AS Status_Count,
    GROUP_CONCAT(
        DISTINCT Order_Status
        ORDER BY Order_Status
        SEPARATOR ' / '
    ) AS Status_Values
FROM bronze.ecommerce_raw
WHERE Order_ID IN (
    'ORD100427',
    'ORD100700',
    'ORD101794',
    'ORD102113',
    'ORD102116'
)
GROUP BY Order_ID
ORDER BY Order_ID;
"""

cursor.execute(conflict_query)

sql_conflict_results = cursor.fetchall()

for row in sql_conflict_results:
    print(row)

cursor.close()
connection.close()

('ORD100427', 2, 2, 'Delivered / Pending')
('ORD100700', 2, 2, 'Delayed / Delivered')
('ORD101794', 2, 2, 'Delivered / Pending')
('ORD102113', 2, 2, 'Delivered / Returned')
('ORD102116', 2, 2, 'Delayed / Delivered')


In [28]:
# Cell 26: Create Silver duplicate exceptions table

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS silver.ecommerce_duplicate_exceptions
    LIKE bronze.ecommerce_raw
""")

print("Silver duplicate exceptions table created or verified.")

cursor.close()
connection.close()

Silver duplicate exceptions table created or verified.


In [29]:
# Cell 27: Create Silver deduplicated candidate table

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS silver.ecommerce_deduplicated
    LIKE bronze.ecommerce_raw
""")

print("Silver deduplicated table created or verified.")

cursor.close()
connection.close()

Silver deduplicated table created or verified.


In [30]:
# Cell 28: Move conflicting orders into Silver exceptions

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

try:
    # Check whether the target table is empty
    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.ecommerce_duplicate_exceptions
    """)

    existing_count = cursor.fetchone()[0]

    if existing_count > 0:
        raise Exception(
            "Exceptions table is not empty. "
            "Insertion cancelled to prevent duplicates."
        )

    # Preserve all original columns and ingestion metadata
    all_columns = business_columns + [
        "ingestion_id",
        "source_file",
        "ingestion_timestamp"
    ]

    column_list = ", ".join(
        f"`{column}`" for column in all_columns
    )

    insert_query = f"""
        INSERT INTO silver.ecommerce_duplicate_exceptions
        ({column_list})
        SELECT {column_list}
        FROM bronze.ecommerce_raw
        WHERE Order_ID IN (
            'ORD100427',
            'ORD100700',
            'ORD101794',
            'ORD102113',
            'ORD102116'
        )
    """

    cursor.execute(insert_query)
    connection.commit()

    print("Conflicting records preserved successfully!")
    print("Records inserted:", cursor.rowcount)

except Exception as error:
    connection.rollback()
    print("Insertion failed:", error)

finally:
    cursor.close()
    connection.close()

Conflicting records preserved successfully!
Records inserted: 10


In [31]:
# Cell 29: Deduplicate records using SQL ROW_NUMBER()

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

try:
    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.ecommerce_deduplicated
    """)

    existing_count = cursor.fetchone()[0]

    if existing_count > 0:
        raise Exception(
            "Silver table already contains records. "
            "Insertion cancelled."
        )

    columns = business_columns + [
        "ingestion_id",
        "source_file",
        "ingestion_timestamp"
    ]

    column_list = ", ".join(
        f"`{column}`" for column in columns
    )

    deduplication_query = f"""
        INSERT INTO silver.ecommerce_deduplicated ({column_list})

        SELECT {column_list}
        FROM (
            SELECT
                b.*,

                ROW_NUMBER() OVER (
                    PARTITION BY
                        {", ".join(f"`{col}`" for col in business_columns)}
                    ORDER BY ingestion_id
                ) AS row_num

            FROM bronze.ecommerce_raw b

            WHERE NOT EXISTS (
                SELECT 1
                FROM silver.ecommerce_duplicate_exceptions e
                WHERE e.ingestion_id = b.ingestion_id
            )

        ) AS ranked_records

        WHERE row_num = 1
    """

    cursor.execute(deduplication_query)
    connection.commit()

    print("Silver deduplication completed!")
    print("Records inserted:", cursor.rowcount)

except Exception as error:
    connection.rollback()
    print("Deduplication failed:", error)

finally:
    cursor.close()
    connection.close()

Silver deduplication completed!
Records inserted: 2995


In [32]:
# Cell 30: Validate Bronze and Silver record counts

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

validation_query = """
SELECT 'Bronze' AS Layer, COUNT(*) AS Records
FROM bronze.ecommerce_raw

UNION ALL

SELECT 'Silver Deduplicated', COUNT(*)
FROM silver.ecommerce_deduplicated

UNION ALL

SELECT 'Silver Exceptions', COUNT(*)
FROM silver.ecommerce_duplicate_exceptions;
"""

cursor.execute(validation_query)

results = cursor.fetchall()

for row in results:
    print(f"{row[0]}: {row[1]}")

cursor.close()
connection.close()

Bronze: 3055
Silver Deduplicated: 2995
Silver Exceptions: 10


In [33]:
# Cell 31: Verify duplicate Order_IDs in Silver candidate

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

cursor.execute("""
    SELECT
        Order_ID,
        COUNT(*) AS Record_Count
    FROM silver.ecommerce_deduplicated
    GROUP BY Order_ID
    HAVING COUNT(*) > 1
""")

remaining_duplicates = cursor.fetchall()

print("Remaining duplicate Order_ID groups:",
      len(remaining_duplicates))

if remaining_duplicates:
    print(remaining_duplicates)

cursor.close()
connection.close()

Remaining duplicate Order_ID groups: 0


In [34]:
# Cell 32: Analyze missing values in Silver candidate

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

df_silver_candidate = pd.read_sql_query(
    """
    SELECT *
    FROM silver.ecommerce_deduplicated
    """,
    engine
)

connection.close()

# Business columns only
df_silver_business = df_silver_candidate[
    business_columns
].copy()

# Treat empty and whitespace-only strings as missing for analysis
df_silver_profile = df_silver_business.replace(
    r"^\s*$",
    pd.NA,
    regex=True
)

silver_missing_profile = pd.DataFrame({
    "Column": business_columns,
    "Missing_Count": df_silver_profile.isna().sum().values,
    "Missing_Percentage": (
        df_silver_profile.isna().mean().values * 100
    ).round(2)
})

silver_missing_profile = silver_missing_profile.sort_values(
    by="Missing_Count",
    ascending=False
)

print("Silver Candidate Records:", len(df_silver_business))
print(
    "Total Missing Cells:",
    df_silver_profile.isna().sum().sum()
)

display(silver_missing_profile)

Silver Candidate Records: 2995
Total Missing Cells: 2446


,Column,Missing_Count,Missing_Percentage
31,Profit_Margin_Pct,239,7.98
28,Net_Sales,238,7.95
29,Estimated_Cost,238,7.95
27,Discount_Amount,238,7.95
30,Profit,238,7.95
16,Customer_Rating,209,6.98
14,Discount_Pct,149,4.97
25,Customer_Satisfaction,149,4.97
3,Customer_Age,120,4.01
21,Customer_Loyalty_Points,119,3.97


In [35]:
# Cell 33: Analyze missing-value patterns in financial columns

financial_columns = [
    "Unit_Price",
    "Gross_Amount",
    "Discount_Pct",
    "Discount_Amount",
    "Net_Sales",
    "Estimated_Cost",
    "Profit",
    "Profit_Margin_Pct"
]

financial_data = df_silver_business[
    financial_columns
].replace(r"^\s*$", pd.NA, regex=True)

financial_missing = financial_data.isna()

print("Missing Values by Financial Column:")
display(financial_missing.sum().to_frame("Missing_Count"))

print("\nFinancial Missing-Value Pattern:")

pattern_summary = (
    financial_missing
    .astype(int)
    .groupby(financial_columns)
    .size()
    .reset_index(name="Record_Count")
    .sort_values("Record_Count", ascending=False)
)

display(pattern_summary)

print("\nRecords with All Financial Columns Missing:",
      financial_missing.all(axis=1).sum())

print("Records with At Least One Financial Value Missing:",
      financial_missing.any(axis=1).sum())

Missing Values by Financial Column:


,Missing_Count
Unit_Price,90
Gross_Amount,90
Discount_Pct,149
Discount_Amount,238
Net_Sales,238
Estimated_Cost,238
Profit,238
Profit_Margin_Pct,239



Financial Missing-Value Pattern:


,Unit_Price,Gross_Amount,Discount_Pct,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct,Record_Count
0,0,0,0,0,0,0,0,0,2756
2,0,0,1,1,1,1,1,1,148
3,1,1,0,1,1,1,1,1,89
1,0,0,0,0,0,0,0,1,1
4,1,1,1,1,1,1,1,1,1



Records with All Financial Columns Missing: 1
Records with At Least One Financial Value Missing: 239


In [36]:
# Cell 34: Detect hidden missing-value representations

missing_tokens = {
    "", "nan", "null", "none", "n/a", "na", "nat"
}

token_summary = []

for column in business_columns:

    normalized = (
        df_silver_business[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    token_count = normalized.isin(missing_tokens).sum()

    if token_count > 0:
        token_summary.append({
            "Column": column,
            "Missing_Token_Count": int(token_count)
        })

token_summary_df = pd.DataFrame(token_summary)

print("Missing-value token summary:")
display(token_summary_df)

Missing-value token summary:


,Column,Missing_Token_Count
0,Customer_Age,120
1,City,75
2,Payment_Method,75
3,Unit_Price,90
4,Discount_Pct,149
5,Shipping_Days,105
6,Customer_Rating,209
7,Customer_Loyalty_Points,119
8,Website_Session_Minutes,75
9,Customer_Satisfaction,149


In [37]:
# Cell 35: Inspect records with missing financial information

financial_missing = financial_data.isna()

affected_records = df_silver_business.loc[
    financial_missing.any(axis=1),
    [
        "Order_ID",
        "Product_Name",
        "Category",
        "Quantity",
        "Unit_Price",
        "Gross_Amount",
        "Discount_Pct",
        "Discount_Amount",
        "Net_Sales",
        "Estimated_Cost",
        "Profit",
        "Profit_Margin_Pct"
    ]
].copy()

print("Affected Records:", len(affected_records))

display(affected_records.head(15))

Affected Records: 239


,Order_ID,Product_Name,Category,Quantity,Unit_Price,Gross_Amount,Discount_Pct,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
1,ORD101072,Shoes,Home Appliances,2,,,28.01,,,,,
22,ORD101810,Chair,Home Appliances,1,5160.28,5160.28,,,,,,
28,ORD102765,Shoes,Furniture,3,,,25.52,,,,,
32,ORD100424,Mixer,Books,5,1602.68,8013.4,,,,,,
34,ORD100823,Headphones,Sports,4,1450.1,5800.4,,,,,,
38,ORD101124,Treadmill,Books,2,402.93,805.86,,,,,,
42,ORD102595,Treadmill,Books,4,1250.58,5002.32,,,,,,
45,ORD101424,Headphones,Books,2,324.54,649.08,,,,,,
52,ORD100026,Rice,Electronics,2,,,22.36,,,,,
71,ORD101361,Novel,Home Appliances,2,,,16.36,,,,,


In [38]:
# Cell 36: Validate financial relationships using complete records

financial_check = df_silver_business.copy()

numeric_columns = [
    "Quantity",
    "Unit_Price",
    "Gross_Amount",
    "Discount_Pct",
    "Discount_Amount",
    "Net_Sales",
    "Estimated_Cost",
    "Profit",
    "Profit_Margin_Pct"
]

for col in numeric_columns:
    financial_check[col] = pd.to_numeric(
        financial_check[col],
        errors="coerce"
    )

# Validate each formula only where its required inputs exist
checks = {
    "Gross = Quantity * Unit Price": [
        "Quantity", "Unit_Price", "Gross_Amount"
    ],
    "Discount = Gross * Discount %": [
        "Gross_Amount", "Discount_Pct", "Discount_Amount"
    ],
    "Net Sales = Gross - Discount": [
        "Gross_Amount", "Discount_Amount", "Net_Sales"
    ],
    "Profit = Net Sales - Cost": [
        "Net_Sales", "Estimated_Cost", "Profit"
    ],
    "Margin = Profit / Net Sales * 100": [
        "Profit", "Net_Sales", "Profit_Margin_Pct"
    ]
}

results = []

for formula, columns in checks.items():

    valid = financial_check[columns].notna().all(axis=1)
    data = financial_check.loc[valid]

    if formula == "Gross = Quantity * Unit Price":
        expected = data["Quantity"] * data["Unit_Price"]
        actual = data["Gross_Amount"]

    elif formula == "Discount = Gross * Discount %":
        expected = data["Gross_Amount"] * data["Discount_Pct"] / 100
        actual = data["Discount_Amount"]

    elif formula == "Net Sales = Gross - Discount":
        expected = data["Gross_Amount"] - data["Discount_Amount"]
        actual = data["Net_Sales"]

    elif formula == "Profit = Net Sales - Cost":
        expected = data["Net_Sales"] - data["Estimated_Cost"]
        actual = data["Profit"]

    else:
        expected = (
            data["Profit"] / data["Net_Sales"] * 100
        )
        actual = data["Profit_Margin_Pct"]

    difference = (expected - actual).abs()

    results.append({
        "Formula": formula,
        "Comparable_Records": len(data),
        "Matching_Records": int((difference <= 0.02).sum()),
        "Mismatching_Records": int((difference > 0.02).sum()),
        "Max_Absolute_Difference": round(
            difference.max(), 4
        ) if len(difference) else None
    })

formula_validation = pd.DataFrame(results)

display(formula_validation)

,Formula,Comparable_Records,Matching_Records,Mismatching_Records,Max_Absolute_Difference
0,Gross = Quantity * Unit Price,2905,2905,0,0.000
1,Discount = Gross * Discount %,2757,2757,0,0.005
2,Net Sales = Gross - Discount,2757,2757,0,0.000
3,Profit = Net Sales - Cost,2757,2757,0,0.000
4,Margin = Profit / Net Sales * 100,2756,2756,0,0.005


In [39]:
# Cell 37: Count records where financial values can be reconstructed

df_check = financial_check.copy()

reconstruction_checks = {
    "Gross_Amount": (
        df_check["Gross_Amount"].isna()
        & df_check["Quantity"].notna()
        & df_check["Unit_Price"].notna()
    ),

    "Discount_Amount": (
        df_check["Discount_Amount"].isna()
        & df_check["Gross_Amount"].notna()
        & df_check["Discount_Pct"].notna()
    ),

    "Net_Sales": (
        df_check["Net_Sales"].isna()
        & df_check["Gross_Amount"].notna()
        & df_check["Discount_Amount"].notna()
    ),

    "Profit": (
        df_check["Profit"].isna()
        & df_check["Net_Sales"].notna()
        & df_check["Estimated_Cost"].notna()
    ),

    "Profit_Margin_Pct": (
        df_check["Profit_Margin_Pct"].isna()
        & df_check["Profit"].notna()
        & df_check["Net_Sales"].notna()
        & df_check["Net_Sales"].ne(0)
    )
}

reconstruction_summary = pd.DataFrame([
    {
        "Column": column,
        "Missing_Values": int(df_check[column].isna().sum()),
        "Recoverable_Values": int(condition.sum()),
        "Not_Recoverable": int(
            df_check[column].isna().sum() - condition.sum()
        )
    }
    for column, condition in reconstruction_checks.items()
])

display(reconstruction_summary)

,Column,Missing_Values,Recoverable_Values,Not_Recoverable
0,Gross_Amount,90,0,90
1,Discount_Amount,238,0,238
2,Net_Sales,238,0,238
3,Profit,238,0,238
4,Profit_Margin_Pct,239,0,239


In [40]:
import os

os.chdir(PROJECT_ROOT)

gitignore_content = """
# Environment and credentials
.env
.env.*
!.env.example

# Python
__pycache__/
*.py[cod]
.venv/
venv/

# Jupyter
.ipynb_checkpoints/

# Logs
logs/
*.log

# IDE
.vscode/
.idea/

# OS
.DS_Store
Thumbs.db
"""

with open(".gitignore", "w") as file:
    file.write(gitignore_content.strip() + "\n")

print(".gitignore created successfully.")

.gitignore created successfully.
